<a href="https://colab.research.google.com/github/ehsanre1376/YouTube-DownLoader-To-Colab/blob/main/TelBot2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# -*- coding: utf-8 -*-
import os
import sys
import logging
import asyncio
import re
import zipfile
import subprocess
import time
import shutil
from pathlib import Path
from telegram import Update, Bot, InlineKeyboardButton, InlineKeyboardMarkup, InputFile
from telegram.ext import Application, CommandHandler, MessageHandler, filters, ContextTypes, CallbackQueryHandler
from telegram.constants import ParseMode, ChatAction
import yt_dlp

# --- Configuration ---
BOT_TOKEN = "7668113099:AAHwpP6FHOlifpSqhFZTj03Zc6r0k2HrR8o" # Your Bot Token Here
DOWNLOAD_BASE_DIR = "youtube_downloads"  # Base directory for downloads
MAX_TELEGRAM_FILE_SIZE_BYTES = 1990 * 1024 * 1024 # 1.9GB (slightly less than 2GB for safety)
SUPPORTED_URL_REGEX = r'(?:https?:\/\/)?(?:www\.)?(?:youtube\.com|youtu\.be)\/(?:watch\?v=|playlist\?list=|embed\/|shorts\/|live\/)?([a-zA-Z0-9_-]+)'

# --- Logging Setup ---
logging.basicConfig(
    format="%(asctime)s - %(name)s - %(levelname)s - %(message)s",
    level=logging.INFO,
    handlers=[
        logging.StreamHandler(sys.stdout) # Log to Colab output
    ]
)
logging.getLogger("httpx").setLevel(logging.WARNING) # Suppress noisy httpx logs
logger = logging.getLogger(__name__)

# --- Helper Functions ---

def sanitize_filename(filename):
    """Removes or replaces characters illegal in filenames."""
    # Remove illegal characters
    sanitized = re.sub(r'[\\/*?:"<>|]', "", filename)
    # Replace multiple spaces with single space
    sanitized = re.sub(r'\s+', ' ', sanitized)
    # Remove leading/trailing whitespace
    sanitized = sanitized.strip()
    # Ensure filename is not empty
    if not sanitized:
        sanitized = "downloaded_video"
    # Limit length if necessary (optional, file systems have limits)
    # max_len = 200
    # if len(sanitized) > max_len:
    #     name, ext = os.path.splitext(sanitized)
    #     sanitized = name[:max_len - len(ext) - 1] + ext
    return sanitized

async def send_typing_action(context: ContextTypes.DEFAULT_TYPE, chat_id: int):
    """Sends typing action periodically."""
    try:
        await context.bot.send_chat_action(chat_id=chat_id, action=ChatAction.TYPING)
    except Exception as e:
        logger.warning(f"Could not send typing action: {e}")

async def send_upload_video_action(context: ContextTypes.DEFAULT_TYPE, chat_id: int):
    """Sends upload video action periodically."""
    try:
        await context.bot.send_chat_action(chat_id=chat_id, action=ChatAction.UPLOAD_VIDEO)
    except Exception as e:
        logger.warning(f"Could not send upload video action: {e}")

async def send_upload_document_action(context: ContextTypes.DEFAULT_TYPE, chat_id: int):
    """Sends upload document action periodically."""
    try:
        await context.bot.send_chat_action(chat_id=chat_id, action=ChatAction.UPLOAD_DOCUMENT)
    except Exception as e:
        logger.warning(f"Could not send upload document action: {e}")

def progress_hook(d, status_message, chat_id, context):
    """yt-dlp progress hook to update Telegram status (can be slow/spammy)."""
    if d['status'] == 'downloading':
        # Potentially update status message here, but be careful not to hit rate limits
        # Example:
        # percentage = d['_percent_str']
        # speed = d['_speed_str']
        # eta = d['_eta_str']
        # logger.info(f"Downloading {d['filename']}: {percentage} at {speed}, ETA: {eta}")
        # This can cause "Message not modified" errors if updated too frequently.
        # A simple periodic "TYPING" or "UPLOADING" action is often better.
        pass
    elif d['status'] == 'finished':
        logger.info(f"Finished downloading {d['filename']}")
    elif d['status'] == 'error':
        logger.error(f"Error downloading {d.get('filename', 'unknown file')}: {d.get('error', 'Unknown error')}")

async def download_video_and_subtitles(url: str, download_path: Path, status_message, chat_id: int, context: ContextTypes.DEFAULT_TYPE):
    """Downloads a single video and its subtitles using yt-dlp."""
    sanitized_title = "downloaded_video" # Default
    downloaded_files = {'video': None, 'subtitles': []}

    try:
        # 1. Get Info First (to get title without downloading yet)
        info_opts = {
            'quiet': True,
            'no_warnings': True,
            'skip_download': True,
            'playlist_items': '1', # Only get info for first item if playlist URL (we handle iteration outside)
        }
        with yt_dlp.YoutubeDL(info_opts) as ydl:
             info = ydl.extract_info(url, download=False)
             # Sanitize title for filename BEFORE download
             # Sometimes 'title' might not be present, fallback needed
             base_title = info.get('title', f'youtube_video_{int(time.time())}')
             sanitized_title = sanitize_filename(base_title)
             logger.info(f"Sanitized video title: {sanitized_title}")


        # 2. Prepare Download Options
        output_template = download_path / f"{sanitized_title}.%(ext)s"
        ydl_opts = {
            'format': 'bestvideo[ext=mp4][height<=1080]+bestaudio[ext=m4a]/best[ext=mp4][height<=1080]/best[ext=mp4]/best',
            'outtmpl': str(output_template),
            'writesubtitles': True,
            'subtitleslangs': ['en', 'fa'], # English and Persian
            'subtitlesformat': 'srt',
            'postprocessors': [{ # Ensure MP4 container if merging
                'key': 'FFmpegVideoConvertor',
                'preferedformat': 'mp4',
            }],
            'merge_output_format': 'mp4', # Ensure merged output is mp4
            'quiet': False, # Show download progress in logs
            'no_warnings': True,
            'noplaylist': True, # Process only single video here
            'progress_hooks': [lambda d: progress_hook(d, status_message, chat_id, context)],
            'ffmpeg_location': '/usr/bin/ffmpeg', # Explicitly set ffmpeg path (usually works on Colab)
            'verbose': False, # Set to True for extreme debugging
            'ignoreerrors': False, # Stop on errors for this single video
            # 'socket_timeout': 30, # Optional: Timeout for network operations
        }

        logger.info(f"Starting download for: {url} with title: {sanitized_title}")
        await send_typing_action(context, chat_id)

        # 3. Perform Download
        try:
            with yt_dlp.YoutubeDL(ydl_opts) as ydl:
                ydl.download([url])
        except yt_dlp.utils.DownloadError as e:
            logger.error(f"yt-dlp download error for {url}: {e}")
            # Check if it's a subtitle error (often non-critical)
            if "subtitles" in str(e).lower():
                 logger.warning(f"Could not download subtitles for {url}, continuing with video.")
                 # Still try to find the video file even if subs failed
            else:
                 await context.bot.edit_message_text(chat_id=chat_id, message_id=status_message.message_id,
                                                    text=f"❌ Failed to download video: {sanitized_title}\nError: {str(e)[:100]}...")
                 return None # Indicate failure
        except Exception as e:
             logger.error(f"Unexpected error during yt-dlp download for {url}: {e}")
             await context.bot.edit_message_text(chat_id=chat_id, message_id=status_message.message_id,
                                                 text=f"❌ An unexpected error occurred during download for: {sanitized_title}")
             return None

        # 4. Find Downloaded Files
        logger.info(f"Searching for downloaded files in: {download_path}")
        video_file = None
        subtitle_files = []
        for filename in os.listdir(download_path):
            full_path = download_path / filename
            logger.debug(f"Checking file: {full_path}")
            # Check if the filename *starts* with the sanitized title
            # This handles cases where yt-dlp might add resolution/format info if title is ambiguous
            if filename.startswith(sanitized_title) and filename.lower().endswith(".mp4"):
                video_file = full_path
                logger.info(f"Found video file: {video_file}")
            elif filename.startswith(sanitized_title) and filename.lower().endswith((".srt", ".en.srt", ".fa.srt")):
                subtitle_files.append(full_path)
                logger.info(f"Found subtitle file: {full_path}")

        if not video_file or not video_file.is_file():
             logger.error(f"Could not find the downloaded MP4 video file starting with '{sanitized_title}' in {download_path}")
             # Fallback: check for *any* mp4 file in the directory if the title match failed
             found_mp4s = list(download_path.glob('*.mp4'))
             if found_mp4s:
                 video_file = found_mp4s[0]
                 logger.warning(f"Using fallback video file: {video_file}")
             else:
                 await context.bot.edit_message_text(chat_id=chat_id, message_id=status_message.message_id,
                                                text=f"❌ Download seemed complete, but couldn't locate the final MP4 file for: {sanitized_title}")
                 return None

        downloaded_files['video'] = video_file
        downloaded_files['subtitles'] = subtitle_files

        logger.info(f"Successfully processed: {sanitized_title}")
        return downloaded_files # Contains Path objects

    except yt_dlp.utils.ExtractorError as e:
        logger.error(f"yt-dlp extractor error for {url}: {e}")
        await context.bot.edit_message_text(chat_id=chat_id, message_id=status_message.message_id,
                                            text=f"❌ Error extracting info from URL. Is it valid?\n{str(e)[:100]}")
        return None
    except Exception as e:
        logger.exception(f"Unexpected error in download_video_and_subtitles for {url}")
        await context.bot.edit_message_text(chat_id=chat_id, message_id=status_message.message_id,
                                            text=f"❌ An unexpected error occurred: {e}")
        return None

async def create_zip_archive(source_dir: Path, zip_filename: Path):
    """Creates a zip archive of the contents of source_dir."""
    logger.info(f"Creating zip archive: {zip_filename} from {source_dir}")
    try:
        with zipfile.ZipFile(zip_filename, 'w', zipfile.ZIP_DEFLATED) as zipf:
            for root, _, files in os.walk(source_dir):
                for file in files:
                    file_path = Path(root) / file
                    arcname = file_path.relative_to(source_dir) # Path inside zip
                    zipf.write(file_path, arcname=arcname)
        logger.info(f"Successfully created zip archive: {zip_filename}")
        return zip_filename
    except Exception as e:
        logger.exception(f"Failed to create zip archive {zip_filename}")
        return None

async def split_and_send_zip(zip_path: Path, chat_id: int, context: ContextTypes.DEFAULT_TYPE, status_message):
    """Splits a large zip file and sends the parts."""
    try:
        zip_dir = zip_path.parent
        base_name = zip_path.stem # Filename without extension

        # Use the 'zip' command-line tool for splitting
        # Example: zip -s 1900m ../split_archive.zip *.mp4 *.srt
        split_size = "1900m" # Slightly less than 2GB
        split_archive_base = zip_dir / f"{base_name}_split" # Base name for split parts

        # Important: The zip command usually creates the split files in the CWD *or* relative to the zip file name provided.
        # Let's ensure it creates them in our target directory.
        # We need to zip the *contents* of the original (now too large) zip, or re-zip the source.
        # Re-zipping with split command is safer.
        # The command needs to be run from *within* the directory containing the files to zip.

        # 1. Get list of files that were originally zipped
        files_to_zip = []
        with zipfile.ZipFile(zip_path, 'r') as zf:
            files_to_zip = zf.namelist()

        if not files_to_zip:
             logger.error("Could not get file list from the large zip for splitting.")
             await context.bot.send_message(chat_id=chat_id, text="❌ Error preparing large file for splitting.")
             return False

        # 2. Delete the large unsplit zip file
        try:
            os.remove(zip_path)
        except OSError as e:
            logger.error(f"Could not remove large zip {zip_path} before splitting: {e}")
            # Proceeding might still work if the zip command overwrites or ignores

        # 3. Run the zip command from the source directory of the original files
        source_dir = Path(f"{DOWNLOAD_BASE_DIR}/{chat_id}_{status_message.message_id}") # Reconstruct source dir path
        cmd = [
            'zip',
            '-s', split_size,              # Split size
            str(split_archive_base),       # Output zip base name (will add .zip, .z01, etc.)
        ] + files_to_zip                   # Add files to include
        logger.info(f"Running split command: {' '.join(cmd)} in directory {source_dir}")

        # Execute command
        process = await asyncio.create_subprocess_exec(
            *cmd,
            stdout=asyncio.subprocess.PIPE,
            stderr=asyncio.subprocess.PIPE,
            cwd=source_dir # Run the command from the directory containing the files
        )
        stdout, stderr = await process.communicate()

        if process.returncode != 0:
            logger.error(f"Zip split command failed with code {process.returncode}")
            logger.error(f"Stderr: {stderr.decode()}")
            logger.error(f"Stdout: {stdout.decode()}")
            await context.bot.edit_message_text(chat_id=chat_id, message_id=status_message.message_id,
                                                text="❌ Failed to split the large archive.")
            return False
        else:
            logger.info(f"Zip split command successful.")
            logger.info(f"Stdout: {stdout.decode()}")

        # 4. Find and send the split files
        split_files = sorted(list(zip_dir.glob(f"{base_name}_split.*"))) # Find .zip, .z01, .z02...
        if not split_files:
             logger.error(f"Could not find split zip files matching {base_name}_split.* in {zip_dir}")
             await context.bot.edit_message_text(chat_id=chat_id, message_id=status_message.message_id,
                                                 text="❌ Splitting seemed to work, but couldn't find the split files.")
             return False

        await context.bot.edit_message_text(chat_id=chat_id, message_id=status_message.message_id,
                                            text=f"📦 Archive split into {len(split_files)} parts. Uploading...")
        await send_upload_document_action(context, chat_id)

        total_parts = len(split_files)
        for i, part_path in enumerate(split_files):
             part_filename = part_path.name
             await context.bot.send_message(chat_id=chat_id, text=f"⬆️ Uploading part {i+1}/{total_parts}: `{part_filename}`", parse_mode=ParseMode.MARKDOWN_V2)
             await send_upload_document_action(context, chat_id) # Keep showing activity
             try:
                 with open(part_path, 'rb') as f:
                     await context.bot.send_document(
                         chat_id=chat_id,
                         document=InputFile(f, filename=part_filename),
                         caption=f"Split archive part {i+1}/{total_parts}. Use a tool like 7-Zip or `zip -s 0 {base_name}_split.zip --out unsplit.zip` to rejoin."
                     )
             except Exception as e:
                 logger.exception(f"Failed to send split zip part: {part_path}")
                 await context.bot.send_message(chat_id=chat_id, text=f"❌ Failed to upload part: {part_filename}\nError: {e}")
                 # Decide if you want to abort here or try sending remaining parts
                 return False # Abort on first failure

        logger.info(f"Finished sending all {len(split_files)} split parts.")
        return True # Indicate success

    except Exception as e:
        logger.exception("Error during zip splitting process")
        await context.bot.send_message(chat_id=chat_id, text=f"❌ An unexpected error occurred during file splitting: {e}")
        return False

async def send_file_robustly(file_path: Path, chat_id: int, context: ContextTypes.DEFAULT_TYPE, caption: str = "", is_video: bool = False):
    """Sends a file (video or document) to Telegram, handling potential errors."""
    if not file_path or not file_path.is_file():
        logger.error(f"File not found for sending: {file_path}")
        return False

    file_size = file_path.stat().st_size
    logger.info(f"Attempting to send {'video' if is_video else 'document'}: {file_path.name} (Size: {file_size / (1024*1024):.2f} MB)")

    if file_size <= 0:
        logger.warning(f"File size is 0 for {file_path.name}, skipping send.")
        return False # Don't send empty files

    # Check Telegram's absolute limit (though splitting handles > 2GB for zips)
    # Individual videos still have the 2GB limit.
    if file_size > MAX_TELEGRAM_FILE_SIZE_BYTES * 1.05: # Use a slight buffer
         logger.warning(f"File {file_path.name} is larger than 2GB ({file_size / (1024*1024):.2f} MB), cannot send directly.")
         await context.bot.send_message(chat_id=chat_id, text=f"⚠️ File `{file_path.name}` is larger than 2GB and cannot be sent directly via Telegram.", parse_mode=ParseMode.MARKDOWN_V2)
         return False # Indicate it wasn't sent

    action = ChatAction.UPLOAD_VIDEO if is_video else ChatAction.UPLOAD_DOCUMENT
    try:
        await context.bot.send_chat_action(chat_id=chat_id, action=action)
        with open(file_path, 'rb') as f:
            input_file = InputFile(f, filename=file_path.name)
            if is_video:
                await context.bot.send_video(chat_id=chat_id, video=input_file, caption=caption, connect_timeout=60, read_timeout=180) # Increased timeouts
            else:
                await context.bot.send_document(chat_id=chat_id, document=input_file, caption=caption, connect_timeout=60, read_timeout=180) # Increased timeouts
        logger.info(f"Successfully sent: {file_path.name}")
        return True
    except Exception as e:
        logger.exception(f"Failed to send file {file_path.name} to chat {chat_id}")
        await context.bot.send_message(chat_id=chat_id, text=f"❌ Failed to upload file: `{file_path.name}`\nError: {str(e)[:100]}...", parse_mode=ParseMode.MARKDOWN_V2)
        return False

def cleanup_directory(directory_path: Path):
    """Removes the specified directory and its contents."""
    try:
        if directory_path.exists() and directory_path.is_dir():
            shutil.rmtree(directory_path)
            logger.info(f"Successfully cleaned up directory: {directory_path}")
        else:
             logger.warning(f"Cleanup directory not found or not a directory: {directory_path}")
    except Exception as e:
        logger.exception(f"Error during cleanup of directory {directory_path}")


# --- Telegram Bot Handlers ---

async def start_command(update: Update, context: ContextTypes.DEFAULT_TYPE) -> None:
    """Sends a welcome message when the /start command is issued."""
    user = update.effective_user
    logger.info(f"User {user.id} ({user.username}) started the bot.")
    keyboard = [
        [InlineKeyboardButton("❓ Help", callback_data='help')],
    ]
    reply_markup = InlineKeyboardMarkup(keyboard)
    await update.message.reply_html(
        rf"👋 Hello {user.mention_html()}!",
        reply_markup=reply_markup
    )
    await asyncio.sleep(0.5) # Slight delay before sending next message
    await update.message.reply_text(
        "I can download YouTube videos and playlists for you!\n\n"
        "Just send me a valid YouTube video or playlist URL.\n\n"
        "✅ Downloads up to 1080p MP4\n"
        "✅ Gets English (en) & Persian (fa) subtitles (if available)\n"
        "✅ Sends individual videos\n"
        "✅ Sends a ZIP archive with all videos & subtitles\n"
        "✅ Handles large archives (> 2GB) by splitting"
    )

async def help_command(update: Update, context: ContextTypes.DEFAULT_TYPE) -> None:
    """Sends help information."""
    logger.info(f"User {update.effective_user.id} requested help.")
    help_text = (
        "ℹ️ **How to Use Me:**\n\n"
        "1.  **Send a URL:** Paste a YouTube video URL (like `https://www.youtube.com/watch?v=...`) or a playlist URL (like `https://www.youtube.com/playlist?list=...`) into the chat.\n\n"
        "2.  **Wait:** I will start downloading the video(s) and subtitles.\n    *   For playlists, I'll send each video as soon as it's ready.\n    *   I'll show status updates.\n\n"
        "3.  **Receive Files:**\n    *   You'll get the video(s) as `.mp4` files.\n    *   You'll also get a `.zip` file containing all videos and `.srt` subtitle files (English and Persian, if found).\n    *   If the ZIP file is very large (over ~1.9GB), I'll split it into parts (`.z01`, `.z02`, etc.) which you'll need to download and rejoin using software like 7-Zip.\n\n"
        "**Features:**\n"
        "- Video Quality: Up to 1080p MP4.\n"
        "- Subtitles: English (en) and Persian (fa).\n"
        "- File Naming: Uses video titles.\n"
        "- Cleanup: Files are automatically deleted from the server after sending.\n\n"
        "Just send me a YouTube link to get started!"
    )
    await update.message.reply_text(help_text, parse_mode=ParseMode.MARKDOWN)

async def help_button(update: Update, context: ContextTypes.DEFAULT_TYPE) -> None:
    """Handles the inline help button."""
    query = update.callback_query
    await query.answer() # Acknowledge button press
    logger.info(f"User {query.from_user.id} pressed help button.")
    # Re-use the help_command logic but send as a new message or edit
    # Sending as new message is simpler here
    await help_command(query, context) # Pass query instead of message to reuse help text

async def process_youtube_link(update: Update, context: ContextTypes.DEFAULT_TYPE) -> None:
    """Handles incoming text messages containing YouTube links."""
    url = update.message.text.strip()
    chat_id = update.effective_chat.id
    user_id = update.effective_user.id

    # Basic URL validation
    if not re.match(SUPPORTED_URL_REGEX, url):
        await update.message.reply_text("⚠️ Please send a valid YouTube video or playlist URL.")
        return

    logger.info(f"User {user_id} sent URL: {url}")
    status_message = await update.message.reply_text("⏳ Processing your link...", reply_to_message_id=update.message.message_id)

    # Create unique download directory for this request
    # Using chat_id and message_id ensures uniqueness per request
    request_id = f"{chat_id}_{status_message.message_id}"
    download_dir = Path(DOWNLOAD_BASE_DIR) / request_id
    try:
        download_dir.mkdir(parents=True, exist_ok=True)
        logger.info(f"Created download directory: {download_dir}")
    except OSError as e:
         logger.error(f"Failed to create download directory {download_dir}: {e}")
         await context.bot.edit_message_text(chat_id=chat_id, message_id=status_message.message_id,
                                             text=f"❌ File system error: Could not create temporary directory.")
         return


    all_downloaded_files = [] # Keep track of all successfully downloaded file Path objects for zipping
    playlist_title = "youtube_playlist" # Default name for zip if it's a playlist

    try:
        # --- Check if Playlist ---
        is_playlist = False
        playlist_entries = []
        try:
            await context.bot.edit_message_text(chat_id=chat_id, message_id=status_message.message_id, text="⏳ Checking URL type (video or playlist)...")
            info_opts = {'extract_flat': 'in_playlist', 'skip_download': True, 'quiet': True} # Fast way to check playlist
            with yt_dlp.YoutubeDL(info_opts) as ydl:
                info = ydl.extract_info(url, download=False)
                if info and 'entries' in info:
                    # It's likely a playlist
                    if info.get('_type') == 'playlist' or len(info['entries']) > 1:
                         is_playlist = True
                         playlist_entries = info.get('entries', [])
                         playlist_title = sanitize_filename(info.get('title', 'youtube_playlist'))
                         logger.info(f"Detected Playlist: '{playlist_title}' with {len(playlist_entries)} potential entries.")
                         await context.bot.edit_message_text(chat_id=chat_id, message_id=status_message.message_id,
                                                             text=f"▶️ Playlist detected: '{playlist_title}'. Starting downloads...")
                    else:
                         # Single video URL that might technically be in a playlist context, treat as single
                         logger.info("Detected single video URL.")
                         await context.bot.edit_message_text(chat_id=chat_id, message_id=status_message.message_id, text="🎬 Single video detected. Starting download...")

                else:
                     # Likely a single video
                     logger.info("Detected single video URL (or failed playlist detection).")
                     await context.bot.edit_message_text(chat_id=chat_id, message_id=status_message.message_id, text="🎬 Single video detected. Starting download...")

        except Exception as e:
            logger.error(f"Error checking URL type: {e}")
            await context.bot.edit_message_text(chat_id=chat_id, message_id=status_message.message_id,
                                                 text=f"❌ Couldn't determine if it's a video or playlist. Error: {e}")
            cleanup_directory(download_dir)
            return

        # --- Download ---
        download_tasks = []
        if is_playlist:
             num_entries = len(playlist_entries)
             await context.bot.edit_message_text(chat_id=chat_id, message_id=status_message.message_id,
                                                 text=f"▶️ Playlist '{playlist_title}': Processing {num_entries} videos...")
             for i, entry in enumerate(playlist_entries):
                 entry_url = entry.get('url')
                 entry_title_guess = entry.get('title', f'video_{i+1}') # Use for status messages
                 if not entry_url:
                     logger.warning(f"Skipping playlist item {i+1}, no URL found.")
                     continue

                 await context.bot.edit_message_text(chat_id=chat_id, message_id=status_message.message_id,
                                                     text=f"📥 Downloading video {i+1}/{num_entries}: '{entry_title_guess[:50]}...'")

                 # Download video
                 download_result = await download_video_and_subtitles(entry_url, download_dir, status_message, chat_id, context)

                 if download_result and download_result['video']:
                     video_path = download_result['video']
                     all_downloaded_files.append(video_path) # Add video to zip list
                     if download_result['subtitles']:
                         all_downloaded_files.extend(download_result['subtitles']) # Add subtitles to zip list

                     # Send individual video immediately
                     await context.bot.edit_message_text(chat_id=chat_id, message_id=status_message.message_id,
                                                          text=f"⬆️ Uploading video {i+1}/{num_entries}: '{video_path.name}'...")
                     await send_file_robustly(video_path, chat_id, context, caption=f"Video {i+1}/{num_entries}: {video_path.stem}", is_video=True)
                     await asyncio.sleep(1) # Small delay between uploads
                 else:
                     logger.warning(f"Failed to download or process video {i+1} ('{entry_title_guess}') from playlist.")
                     # Optionally notify user about the specific failure
                     await context.bot.send_message(chat_id=chat_id, text=f"⚠️ Failed to download video {i+1} ('{entry_title_guess[:50]}...')")
                     # Continue with the next video in the playlist

                 # Update status after each video in playlist
                 await context.bot.edit_message_text(chat_id=chat_id, message_id=status_message.message_id,
                                                     text=f"▶️ Playlist '{playlist_title}': Processed {i+1}/{num_entries} videos...")

        else: # Single Video
            download_result = await download_video_and_subtitles(url, download_dir, status_message, chat_id, context)
            if download_result and download_result['video']:
                video_path = download_result['video']
                all_downloaded_files.append(video_path)
                if download_result['subtitles']:
                     all_downloaded_files.extend(download_result['subtitles'])

                # Send the single video
                await context.bot.edit_message_text(chat_id=chat_id, message_id=status_message.message_id,
                                                     text=f"⬆️ Uploading video: '{video_path.name}'...")
                await send_file_robustly(video_path, chat_id, context, caption=f"{video_path.stem}", is_video=True)
            else:
                logger.error(f"Failed to download single video: {url}")
                await context.bot.edit_message_text(chat_id=chat_id, message_id=status_message.message_id,
                                                     text="❌ Failed to download the video.")
                cleanup_directory(download_dir)
                return


        # --- Create and Send ZIP Archive ---
        if not all_downloaded_files:
            logger.warning("No files were successfully downloaded to zip.")
            await context.bot.edit_message_text(chat_id=chat_id, message_id=status_message.message_id,
                                                 text="😕 No videos could be successfully downloaded.")
            cleanup_directory(download_dir)
            return

        await context.bot.edit_message_text(chat_id=chat_id, message_id=status_message.message_id,
                                             text="📦 Creating ZIP archive with all downloaded files...")
        await send_typing_action(context, chat_id)

        # Define zip filename based on playlist title or first video title
        zip_base_name = playlist_title if is_playlist else all_downloaded_files[0].stem
        zip_filename = download_dir.parent / f"{sanitize_filename(zip_base_name)}.zip" # Place zip outside the download dir

        zip_path = await create_zip_archive(download_dir, zip_filename)

        if not zip_path or not zip_path.exists():
             logger.error("Failed to create zip archive.")
             await context.bot.edit_message_text(chat_id=chat_id, message_id=status_message.message_id,
                                                 text="❌ Failed to create the ZIP archive.")
             cleanup_directory(download_dir)
             return

        # --- Check ZIP Size and Send ---
        zip_size = zip_path.stat().st_size
        logger.info(f"Zip file created: {zip_path.name}, Size: {zip_size / (1024*1024):.2f} MB")

        if zip_size > MAX_TELEGRAM_FILE_SIZE_BYTES:
            await context.bot.edit_message_text(chat_id=chat_id, message_id=status_message.message_id,
                                                text=f"📦 ZIP archive is large ({zip_size / (1024*1024):.2f} MB). Splitting into parts...")
            await send_typing_action(context, chat_id)

            # Ensure 'zip' utility is available (needed for splitting)
            try:
                 process_zip_check = await asyncio.create_subprocess_exec('zip', '-v', stdout=asyncio.subprocess.PIPE, stderr=asyncio.subprocess.PIPE)
                 await process_zip_check.communicate()
                 if process_zip_check.returncode != 0:
                    raise FileNotFoundError
                 logger.info("'zip' command found.")
            except FileNotFoundError:
                 logger.error("'zip' command-line utility not found. Cannot split large archives.")
                 logger.info("Attempting to install 'zip'...")
                 await context.bot.edit_message_text(chat_id=chat_id, message_id=status_message.message_id, text="⚠️ 'zip' tool not found. Trying to install it for splitting (may take a moment)...")
                 # Attempt to install zip (works on Debian/Ubuntu based systems like Colab)
                 process_install = await asyncio.create_subprocess_exec('apt-get', 'update', '-y', stdout=asyncio.subprocess.PIPE, stderr=asyncio.subprocess.PIPE)
                 await process_install.communicate()
                 process_install = await asyncio.create_subprocess_exec('apt-get', 'install', '-y', 'zip', stdout=asyncio.subprocess.PIPE, stderr=asyncio.subprocess.PIPE)
                 stdout_install, stderr_install = await process_install.communicate()
                 if process_install.returncode == 0:
                      logger.info("'zip' installed successfully.")
                      await context.bot.edit_message_text(chat_id=chat_id, message_id=status_message.message_id, text="✅ 'zip' installed. Proceeding with splitting...")
                 else:
                      logger.error(f"Failed to install 'zip'. Stderr: {stderr_install.decode()}")
                      await context.bot.edit_message_text(chat_id=chat_id, message_id=status_message.message_id, text="❌ Failed to install 'zip'. Cannot split large archive. Try downloading individual files.")
                      # Try sending the original large zip? No, Telegram will reject. Just clean up.
                      cleanup_directory(download_dir)
                      if zip_path.exists(): os.remove(zip_path)
                      return

            # Proceed with splitting
            split_success = await split_and_send_zip(zip_path, chat_id, context, status_message)
            if not split_success:
                logger.error("Failed to split and send the zip archive.")
                # Status message might have been updated by split function
            else:
                await context.bot.send_message(chat_id=chat_id, text="✅ All split archive parts sent.")

        else: # Zip file is within size limits
            await context.bot.edit_message_text(chat_id=chat_id, message_id=status_message.message_id,
                                                text=f"⬆️ Uploading ZIP archive: '{zip_path.name}' ({zip_size / (1024*1024):.2f} MB)")
            await send_file_robustly(zip_path, chat_id, context, caption=f"ZIP archive: {zip_base_name}")
            await context.bot.send_message(chat_id=chat_id, text="✅ ZIP archive sent.")


        # --- Final Status & Cleanup ---
        await context.bot.edit_message_text(chat_id=chat_id, message_id=status_message.message_id,
                                             text="✅ Processing complete!")

    except yt_dlp.utils.DownloadError as e:
        logger.exception(f"A critical download error occurred for URL: {url}")
        await context.bot.edit_message_text(chat_id=chat_id, message_id=status_message.message_id, text=f"❌ A critical download error occurred:\n`{str(e)}`", parse_mode=ParseMode.MARKDOWN_V2)
    except Exception as e:
        logger.exception(f"An unexpected error occurred processing URL: {url}")
        await context.bot.edit_message_text(chat_id=chat_id, message_id=status_message.message_id, text=f"❌ An unexpected error occurred: {str(e)}")
    finally:
        # --- Cleanup ---
        logger.info(f"Starting cleanup for request {request_id}")
        # Remove the specific download directory
        cleanup_directory(download_dir)
        # Remove the main zip file if it exists
        if 'zip_path' in locals() and zip_path.exists():
            try:
                os.remove(zip_path)
                logger.info(f"Removed main zip file: {zip_path}")
            except OSError as e:
                logger.error(f"Error removing main zip file {zip_path}: {e}")
        # Remove split zip files if they exist
        if 'zip_base_name' in locals() and 'zip_dir' in locals() and is_playlist: # or check if zip_size > limit
           split_files = list(zip_dir.glob(f"{sanitize_filename(zip_base_name)}_split.*"))
           for sf in split_files:
               try:
                   os.remove(sf)
                   logger.info(f"Removed split zip part: {sf}")
               except OSError as e:
                   logger.error(f"Error removing split zip part {sf}: {e}")

        logger.info(f"Cleanup finished for request {request_id}")
        # Optional: Send a final confirmation after cleanup
        # await context.bot.send_message(chat_id=chat_id, text="🧹 Temporary files cleaned up.")


# --- Main Bot Function ---

def main() -> None:
    """Start the bot."""
    # Create the Application and pass it your bot's token.
    application = Application.builder().token(BOT_TOKEN).connect_timeout(30).read_timeout(90).build() # Increased timeouts

    # Create download base directory if it doesn't exist
    Path(DOWNLOAD_BASE_DIR).mkdir(parents=True, exist_ok=True)

    # --- Register Handlers ---
    # On different commands - answer in Telegram
    application.add_handler(CommandHandler("start", start_command))
    application.add_handler(CommandHandler("help", help_command))

    # Handler for the inline help button
    application.add_handler(CallbackQueryHandler(help_button, pattern='^help$'))

    # On non-command textual messages - process the YouTube link
    application.add_handler(MessageHandler(filters.TEXT & ~filters.COMMAND & filters.Regex(SUPPORTED_URL_REGEX), process_youtube_link))

    # Optional: Handler for messages that are text but not a valid youtube link
    application.add_handler(MessageHandler(filters.TEXT & ~filters.COMMAND & ~filters.Regex(SUPPORTED_URL_REGEX), lambda update, context: update.message.reply_text("Hmm, that doesn't look like a YouTube URL. Send /help for instructions.")))

    # Run the bot until the user presses Ctrl-C
    logger.info("Bot starting polling...")
    application.run_polling(allowed_updates=Update.ALL_TYPES)


if __name__ == "__main__":
    # --- Installation Check (Colab specific) ---
    print("Checking/Installing required libraries...")
    try:
        import telegram
        import yt_dlp
        print("python-telegram-bot and yt-dlp seem installed.")
    except ImportError:
        print("Installing python-telegram-bot and yt-dlp...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "python-telegram-bot[job-queue]", "yt-dlp"])
        print("Installation complete. Please RESTART the Colab runtime if you saw install messages, then run this cell again.")
        # Exit here if installs happened, user needs to restart
        # sys.exit("Restart runtime required after installation.") # Uncomment this line if you want to force exit

    # Check for ffmpeg (usually present on Colab)
    print("Checking for ffmpeg...")
    if shutil.which("ffmpeg"):
        print("ffmpeg found.")
    else:
        print("ffmpeg not found. yt-dlp might have issues merging formats.")
        # You could attempt installation: !apt-get update && apt-get install -y ffmpeg
        # But it's better to let the user know or rely on yt-dlp's fallback.

    # Check for zip utility (needed for splitting)
    print("Checking for zip utility...")
    if shutil.which("zip"):
        print("zip utility found.")
    else:
        print("zip utility not found. Attempting to install (needed for splitting large archives)...")
        try:
            subprocess.check_call(["apt-get", "update", "-y"])
            subprocess.check_call(["apt-get", "install", "-y", "zip"])
            print("zip utility installed successfully.")
        except Exception as e:
            print(f"Failed to install zip utility: {e}. Splitting large files (>2GB) will fail.")

    print("--- Setup Checks Complete ---")
    main()